# Agente Connect-4: **Minimaxer**
**Estrategia:** Minimax Alpha-Beta

## 1. Descripción del Agente

### Idea principal
Minimaxer implementa el algoritmo Minimax con usando el metodo Alpha-Beta, un algoritmo usado de forma frecuente para juegos que se basen en turnos como ajedrez o en este caso Connect-4. En cada turno el agente construye mentalmente un árbol de juego hasta n niveles de profundidad, asumiendo que el oponente también jugará de forma óptima. Alpha-Beta elimina ramas que no influyen en la decisión final para ahorrar recursos y tiempo de procesamiento sin sacrificar recompensas importante y la calidad de las decisiones.

### Diferencia vs otros agentes del grupo
No se todavia XD

### Componentes del agente

| Componente | Rol |
|---|---|
| score_window(window, player) | Puntúa una ventana de 4 celdas: +5 si hay 3 propias + 1 vacía, −4 si hay 3 rivales + 1 vacía |
| heuristic(board, player) | Suma puntajes de todas las ventanas horizontales, verticales y diagonales + preferencia por columna central |
| minimax(board, depth, α, β, max, player) | Búsqueda recursiva con poda alfa-beta. Alterna entre nodo maximizador (propio) y minimizador (oponente) |
| act(board) | Punto de entrada: detecta color propio, aplica corto-circuito para victoria/bloqueo inmediato, luego llama minimax |
| depth | Parámetro numérico configurable: controla cuántos movimientos al futuro se simula |

### Flujo de decisión

act(board):
  1. Detectar color propio (rojo=-1 o amarillo=1) contando fichas
  2. ¿Puedo ganar en 1 movimiento? → jugar ahí  (corto-circuito)
  3. ¿El rival gana en 1 movimiento? → bloquearlo (corto-circuito)
  4. Para cada columna válida:
       score ← minimax(simulación, depth-1, −∞, +∞, minimizando)
  5. Retornar columna con mayor score


In [ ]:
import sys, os, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

notebook_dir = os.path.abspath('')
tournament_dir = os.path.abspath(os.path.join(notebook_dir, '..', '..'))
if tournament_dir not in sys.path:
    sys.path.insert(0, tournament_dir)
os.chdir(tournament_dir)

from connect4.utils import find_importable_classes
from connect4.policy import Policy
from connect4.connect_state import ConnectState

participants = find_importable_classes('groups', Policy)
Minimaxer = participants['Group D']
Random    = participants['Group A']
print('Agentes:', list(participants.keys()))

def play_game(cls_r, cls_y, kw_r={}, kw_y={}):
    r, y = cls_r(**kw_r), cls_y(**kw_y)
    r.mount(); y.mount()
    state = ConnectState()
    while not state.is_final():
        policy = r if state.player == -1 else y
        state = state.transition(int(policy.act(state.board)))
    return state.get_winner()

def run_series(cls_a, cls_b, n, kw_a={}, kw_b={}):
    w, l, d = 0, 0, 0
    for _ in range(n):
        r = play_game(cls_a, cls_b, kw_a, kw_b)
        if r == -1: w += 1
        elif r == 1: l += 1
        else: d += 1
    return w, l, d

print('Setup listo.')

---
## 2. Análisis de Desempeño
### 2.1 Minimaxer vs Agente Aleatorio — ambos colores

In [ ]:
N, DEPTH = 30, 4

w_r, l_r, d_r = run_series(Minimaxer, Random, N, {'depth': DEPTH})
l_y, w_y, d_y = run_series(Random, Minimaxer, N, {}, {'depth': DEPTH})

print(f'Minimaxer (Rojo)     vs Aleatorio: W={w_r}  L={l_r}  D={d_r}  WR={w_r/N:.0%}')
print(f'Minimaxer (Amarillo) vs Aleatorio: W={w_y}  L={l_y}  D={d_y}  WR={w_y/N:.0%}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (title, w, l, d) in zip(axes, [
    (f'Minimaxer como Rojo\nvs. Agente Aleatorio (N={N})', w_r, l_r, d_r),
    (f'Minimaxer como Amarillo\nvs. Agente Aleatorio (N={N})', w_y, l_y, d_y),
]):
    bars = ax.bar(['Victorias','Derrotas','Empates'], [w,l,d],
                  color=['#2ecc71','#e74c3c','#bdc3c7'], edgecolor='#2c3e50', lw=1.2, width=0.5)
    for bar, v in zip(bars, [w,l,d]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{v}\n({v/N:.0%})', ha='center', fontsize=12, fontweight='bold')
    ax.axhline(N*0.5, color='#2980b9', ls='--', lw=1.5, label='Umbral mínimo (50%)')
    ax.set_ylim(0, N+12); ax.set_ylabel('Número de partidos', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True)
fig.suptitle(f'Minimaxer (depth={DEPTH}) vs Agente Aleatorio', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('groups/Group D/fig_vs_random.png', dpi=140, bbox_inches='tight')
plt.show()
print('Figura guardada.')

El minimaxer gana todos sus partidos contra un agente aleatorio con ambos colores, dandole un buen punto de aprtida como un algoritmo efectivo y versatil en diferentes estados de juego iniciales

### 2.2 Mirrormatch

In [ ]:
N_self = 20
w_sr, l_sr, d_s = run_series(Minimaxer, Minimaxer, N_self, {'depth': DEPTH}, {'depth': DEPTH})
print(f'Auto-juego N={N_self}: Rojo={w_sr} ({w_sr/N_self:.0%})  Amarillo={l_sr} ({l_sr/N_self:.0%})  Empates={d_s}')

fig, ax = plt.subplots(figsize=(6.5, 4.5))
vals = [w_sr, l_sr, d_s]
bars = ax.bar(['Rojo gana','Amarillo gana','Empates'], vals,
              color=['#e74c3c','#f39c12','#bdc3c7'], edgecolor='#2c3e50', lw=1.2, width=0.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{v}\n({v/N_self:.0%})', ha='center', fontsize=13, fontweight='bold')
ax.set_title(f'Auto-juego: Minimaxer vs Minimaxer\n(depth={DEPTH}, N={N_self})',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Número de partidos', fontsize=11); ax.set_ylim(0, N_self+5)
ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('groups/Group D/fig_self_play.png', dpi=140, bbox_inches='tight')
plt.show()

En un mirror match, el primer jugador (rojo) suele tener una ventaja significativa, el agente que ocupa el centro primero controla el tablero. Este es el principal cuello de botella identificado.

---
## 3. Propuestas de Mejora

### Cuello de botella 1: Ventaja estructural del primer jugador
El Rojo gana el 100% en auto-juego. La heurística actual no distingue entre colores — usa los mismos pesos para Rojo y Amarillo — lo que no compensa la ventaja del primer movimiento.

**Mejora:** Añadir pesos asimétricos por color en la función heurística: penalizar la columna central para el jugador que mueve primero e incentivar amenazas en filas superiores para el segundo. Esto equilibraría el auto-juego sin afectar la dominancia contra el aleatorio.
